I will clean this to make a demonstration for 1D, 2D+ solvers

In [1]:
using Pkg
using Metal

cd(@__DIR__)
Pkg.activate("../")
ParamFile = "../config/testparam.csv"  # maybe GeoPoints and planet1D should be fusioned

# batchGPU should be at this level (I have not made it as a module yet, since the choice of Metal/CUDA should be done in a manual way)
include("../src/batchFiles/batchGPU.jl")


include("../src/commonBatchs.jl")
include("../src/planet1D.jl")
include("../src/GeoPoints.jl")

include("../src/flexOPT.jl")

using .commonBatchs, .planet1D, .GeoPoints, .flexOPT

  Activating 

devs = Metal.devices() = Metal.MTL.MTLDeviceInstance[Metal.MTL.MTLDeviceInstance (object of type AGXG13XDevice)]

project at `~/Documents/Github/flexOPT`



→ Using Metal backend (1 device(s))
Selected backend type: MetalBackend


# model construction

In [ ]:
using Symbolics,CairoMakie
CairoMakie.activate!()
numPointsX = collect(2:2)
logsOfHinverse = [1.0*i for i in 0:4]

cases=[]
#prefix="B"*string(tmpOrderBspace)*"_"*"w"*string(tmpWorderBspace)*"_"*string(tmpSupplementaryOrder)*"_"
prefix=""
L = 10.0*π
cases = push!(cases,(name=prefix*"λ_2",T=cos(x),β= 1))

@variables x
∂ = Differential(x)


misfit = Array{Float64,3}(undef,length(logsOfHinverse),length(cases),length(numPointsX))


fig = Figure()
ax = Axis(fig[1, 1], xlabel="x", ylabel="model")
iH=5
for iCase ∈ eachindex(cases)
    name,_,β = cases[iCase]
    ΔxTry = exp(-logsOfHinverse[iH])
    Nx = Int(L÷ΔxTry) +1
    Δx = L/(Nx-1)
    X = [Δx * (i-1) for i ∈ range(1,Nx)]
    model=[Symbolics.value(substitute(β,Dict(x=>X[i]))) for i ∈ range(1,Nx)]
    lines!(ax, X, model)
end
fig
Nx=nothing
Δx=nothing
iH=nothing
modelFamily=Array{Any,3}(undef,length(logsOfHinverse),length(cases),length(numPointsX))
forceFamily=Array{Any,3}(undef,length(logsOfHinverse),length(cases),length(numPointsX))
for iPointsUsed ∈ eachindex(numPointsX), iCase ∈ eachindex(cases), iH ∈ eachindex(logsOfHinverse)
    name,T,β = cases[iCase]
    iExperiment = (iH=iH,iCase=iCase,iPointsUsed=iPointsUsed)

    ΔxTry = exp(-logsOfHinverse[iH])
    Nx = Int(L÷ΔxTry) +1
    Δx = L/(Nx-1)
    X = [Δx * (i-1) for i ∈ range(1,Nx)]
    modelName = name*string(Nx)
    models=[]
    model=[Symbolics.value(substitute(β,Dict(x=>X[i]))) for i ∈ range(1,Nx)]
    models=push!(models, model)
    modelPoints = (Nx)
    
    modelFamily[iH,iCase,iPointsUsed]=tmpModel
    q = mySimplify(β*∂(T))
    qₓ = mySimplify(∂(q))
    symbols=(Nx=Nx,Δx=Δx,X=X,q=q,qₓ=qₓ,T=T,β=β)
    tmpModel = (models=models, modelName=modelName, modelPoints=modelPoints,Δ=(Δx),symbols=symbols)
    force = [Symbolics.value(substitute(qₓ,Dict(x=>X[i]))) for i ∈ range(1,Nx)]
    sourceFull=reshape(force,Nx,1,1)
    forceFamily[iH,iCase,iPointsUsed]=sourceFull
   
end


    

# input parameters

In [3]:

famousEquationType="1DpoissonHetero" #GT98
Δ = (1.0)

1.0

In [4]:


# orders: -1 -> indicator function, 0 -> box car, >=1 -> B-spline

orderBtime=1
orderBspace=1

pointsInSpace=3
pointsInTime=3

# the order of errors to be controlled
supplementaryOrder=2

# new parameters for interpolated Taylor expansion μ for field

fieldItpl = (ptsSpace = 1,ptsTime = 1,offsetSpace=1,offsetTime=1,YorderBspace=1,YorderBtime=1) #offsetSpace and offsetTime ∈ z 
# μ points should be distributed from y_min+offset*Δy to y_max-offset*Δy offset can be negative too


# new parameters for interpolated Taylor expansion μᶜ for material
materItpl = (ptsSpace = 1,ptsTime = 1,offsetSpace=1,offsetTime=1,YorderBspace=1,YorderBtime=1)



(ptsSpace = 1, ptsTime = 1, offsetSpace = 1, offsetTime = 1, YorderBspace = 1, YorderBtime = 1)

In [5]:
concreteParametersForOPTConstruction = @strdict famousEquationType Δ orderBtime orderBspace pointsInSpace pointsInTime supplementaryOrder fieldItpl materItpl


Dict{String, Any} with 9 entries:
  "fieldItpl"          => (ptsSpace = 1, ptsTime = 1, offsetSpace = 1, offsetTi…
  "Δ"                  => 1.0
  "supplementaryOrder" => 2
  "materItpl"          => (ptsSpace = 1, ptsTime = 1, offsetSpace = 1, offsetTi…
  "orderBspace"        => 1
  "orderBtime"         => 1
  "famousEquationType" => "1DpoissonHetero"
  "pointsInSpace"      => 3
  "pointsInTime"       => 3

In [6]:
optRec=myProduceOrLoad(makeOPTsemiSymbolic,concreteParametersForOPTConstruction,"semiSymbolic")

Dict{String, Any} with 2 entries:
  "gitcommit" => "c69609f9c80755006ae856b105881f4abe7a2a63-dirty"
  "recette"   => (lhs = (Ajiννᶜ = Num[0.5κ₁ + 0.5κ₂; -0.5κ₁ - κ₂ - 0.5κ₃; 0.5κ₂…

In [7]:
iExperiment=(iH=1,iCase=1,iPointsUsed=1)

(iH = 1, iCase = 1, iPointsUsed = 1)

In [8]:
params=@strdict optRec=optRec modelFam=modelFamily[iExperiment.iH,iExperiment.iCase,iExperiment.iPointsUsed] absorbingBoundaries=nothing maskedRegionInSpace=nothing

Dict{String, Any} with 4 entries:
  "absorbingBoundaries" => nothing
  "optRec"              => Dict{String, Any}("gitcommit"=>"c69609f9c80755006ae8…
  "maskedRegionInSpace" => nothing
  "modelFam"            => (models = Any[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  1, 1…

In [9]:
numOpt=numericalOperatorConstruction(params)

size(models[iVar]) = (32,)
newCoords = expandVectors(size(models[iVar]), CartesianDependency) = [32, 1]
ModelPoints[:, iVar] = newCoords = [32, 1]
tmpModel = reshape(models[iVar], newCoords...) = [1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1;;]
Models[iVar] = tmpModel = [1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1;;]
(Models, ModelPoints) = (Any[[1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1;;]], [32; 1;;])
size(models[iVar]) = (32,)
newCoords = expandVectors(size(models[iVar]), CartesianDependency) = [32, 1]
ModelPoints[:, iVar] = newCoords = [32, 1]
tmpModel = reshape(models[iVar], newCoords...) = [1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1;;]
Models[iVar] = tmpModel = [1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1;;]
(Models, ModelPoint

Dict{String, @NamedTuple{costfunctions::Matrix{Num}, fieldLHS::Matrix{Any}, fieldRHS::Matrix{Any}, champsLimité::Matrix{Any}}} with 1 entry:
  "numOperators" => (costfunctions = [-2.0left_1_t_1₁ + left_1_t_1₂ - 0.834023r…

In [10]:
numOps = numOpt["numOperators"]
prepared = prepareNumericalOperators(numOps)
sourceFull=forceFamily[iExperiment.iH,iExperiment.iCase,iExperiment.iPointsUsed]
syntheticData = timeMarchingSchemePrepared(
    prepared,
    1,
    modelFamily[iExperiment.iH,iExperiment.iCase,iExperiment.iPointsUsed].Δ,
    modelFamily[iExperiment.iH,iExperiment.iCase,iExperiment.iPointsUsed].modelName;
    videoMode=false,
    sourceType="Explicit",
    sourceFull=sourceFull,
    iExperiment=iExperiment,
)
syntheticData=reduce(vcat,syntheticData)
name,T,β = cases[iExperiment.iCase]
            analyticalData = [Symbolics.value(substitute(T,Dict(x=>X[i]))) for i ∈ range(1,Nx)]

            fig =Figure()
            ax=Axis(fig[1,1]; title="Comparison for h=$Δx, model=$(cases[iCase].name), points=1+$pointsInSpace")
            lines!(ax,X,analyticalData,color=:blue)
            scatter!(ax,X,syntheticData,color=:red,marker=:circle)        
            #axislegend(ax)
            display(fig)
            
            #save("plot_$iH_$iCase_$iPointsUsed.png",fig)
            #sleep(0.5)
            #misfit[iH,iCase,iPointsUsed] = norm(syntheticData-analyticalData)


UndefVarError: UndefVarError: `X` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [11]:
Nt = 1
            
            
syntheticData=timeMarchingScheme(numOpt, Nt, Δnum,modelName;videoMode=false,sourceType="Explicit",sourceFull=sourceFull,iExperiment=iExperiment)
syntheticData=reduce(vcat,syntheticData)
analyticalData = [Symbolics.value(substitute(u,Dict(x=>X[i]))) for i ∈ range(1,Nx)]

UndefVarError: UndefVarError: `timeMarchingScheme` not defined in `Main`
Suggestion: check for spelling errors or missing imports.